# Data Discovery & Quality Audit (Exploração Inicial)
Projeto: Retail Analytics Lakehouse (PySpark & Delta Lake)
Autor: Eduardo Bello

Este notebook contém o fluxo exploratório de inspeção e diagnóstico de dados brutos (Raw Layer). O objetivo é identificar anomalias estruturais, incompatibilidades de schema e validar os Quality Gates antes de refatorar a lógica para os módulos de produção em src/.

In [1]:
# [1. SETUP E SESSÃO SPARK]
import yaml
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Carregando configurações do settings.yaml
config_path = Path("../config/settings.yaml") if Path("../config/settings.yaml").exists() else Path("config/settings.yaml")
with open(config_path, "r", encoding="utf-8") as f:
    settings = yaml.safe_load(f)

spark = SparkSession.builder \
    .appName("pyspark-retail-discovery") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()


"""
spark = SparkSession.builder \
    .appName(settings["spark"]["app_name"] + "-discovery") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()
"""
print("[INFO] Sessão Spark inicializada para exploração!")

KeyboardInterrupt: 

## 2. Ingestão Simples dos Arquivos Brutos (Sem Schemas Estritos)

In [ ]:
# [2. LEITURA EXPLORATÓRIA DAS FONTES]
raw_dir = settings["paths"]["raw_data"]

df_sales_raw = spark.read.option("header", "true").csv(f"{raw_dir}/sales.csv")
df_users_raw = spark.read.option("header", "true").csv(f"{raw_dir}/users.csv")

print("--- Schema Bruto: Sales ---")
df_sales_raw.printSchema()
df_sales_raw.show(5)

print("--- Schema Bruto: Users ---")
df_users_raw.printSchema()
df_users_raw.show(5)

## 3. Diagnóstico do JSON Malformado de Produtos

In [ ]:
# [3. INSPEÇÃO E NORMALIZAÇÃO DO PRODUCTS.JSON]
products_file = Path(f"{raw_dir}/products.json")

# Leitura bruta do conteúdo de texto para diagnosticar o problema dos colchetes
raw_text = products_file.read_text(encoding="utf-8").strip()
print(f"Primeiros 100 caracteres do JSON bruto:\n{raw_text[:100]}")

# Aplicação da regra de sanitização caso o array não esteja fechado
if not raw_text.startswith("["):
    sanitized_text = "[" + raw_text.rstrip(",") + "]"
    products_file.write_text(sanitized_text, encoding="utf-8")
    print("[FIX APORTADO] Colchetes adicionados ao JSON com sucesso!")

# Validação da leitura pós-correção
df_products_raw = spark.read.option("multiline", "true").json(str(products_file))
df_products_raw.printSchema()
df_products_raw.show(5)

In [ ]:
## 4. Testes de Quality Gates (Asserts) e Integridade Referencial

In [ ]:
# [4. VALIDAÇÃO DE QUALIDADE DE DADOS]
n_sales = df_sales_raw.count()
dup_sales = n_sales - df_sales_raw.select("sale_id").distinct().count()
null_keys = df_sales_raw.filter(F.col("product_id").isNull() | F.col("client_id").isNull()).count()
orphan_products = df_sales_raw.join(df_products_raw, "product_id", "left_anti").count()

print(f"Total de vendas: {n_sales}")
print(f"Duplicatas em sale_id: {dup_sales}")
print(f"Chaves nulas (product_id/client_id): {null_keys}")
print(f"Produtos órfãos nas vendas: {orphan_products}")

# Validação via Asserts para simular a trava de produção
assert dup_sales == 0 and null_keys == 0, "[FALHA DE DATA QUALITY] Vendas contêm duplicatas ou chaves nulas!"
assert orphan_products == 0, "[FALHA DE DATA QUALITY] Existem produtos órfãos nas vendas!"
print("[SUCESSO] Todos os Quality Gates foram aprovados na exploração!")